# Classical versus simulated neutral-atom association

This notebook is only an interface: every algorithm is imported from the tested package. We initialize one retained track, prepare an ambiguous next frame once, visualize its conflict graph, and hand the identical frozen inputs to the exact classical and QuTiP backends.

The explicit stage order is `predict -> gate -> likelihood -> candidate filter -> graph encode -> cluster -> solve -> Bayesian update -> track filter`. Calling `compare_prepared()` is read-only; only `advance()` changes tracking state.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from neutral_atom_mht.backends.classical import ClassicalBackend
from neutral_atom_mht.backends.neutral_atom import NeutralAtomBackend
from neutral_atom_mht.graph import save_graph_visualization
from neutral_atom_mht.tracking import (
    BayesianConfig, FilterConfig, GateConfig, Observation,
    TrackingConfig, TrackingInterface,
)

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

In [ ]:
config = TrackingConfig(
    seconds_per_frame=1.0,
    initial_velocity_std=2.0,
    filtering=FilterConfig(acceleration_std=0.1, minimum_posterior=1e-4),
    gating=GateConfig(mahalanobis_sq=20.0),
    bayesian=BayesianConfig(clutter_spatial_density=1e-4),
)
tracker = TrackingInterface(config)
classical = ClassicalBackend(maximum_nodes=20)
quantum = NeutralAtomBackend(maximum_simulation_atoms=6)

In [ ]:
initial = (Observation(frame=0, observation_id=1, x=0.0, y=0.0),)
initialized = tracker.step(frame=0, observations=initial, backend=classical)
[(track.track_id, track.position, track.posterior_probability) for track in initialized.tracks]

In [ ]:
ambiguous = (
    Observation(frame=1, observation_id=1, x=0.25, y=0.0),
    Observation(frame=1, observation_id=2, x=1.25, y=0.0),
)
prepared = tracker.prepare(frame=1, observations=ambiguous)
{
    "stage_order": prepared.stage_order,
    "gated_pairs": [(g.track_id, g.observation_id) for g in prepared.gated_associations],
    "weights": {h.hypothesis_id: h.weight for h in prepared.hypotheses},
    "graph_fingerprint": prepared.graph.fingerprint,
    "clusters": [cluster.node_ids for cluster in prepared.clusters],
}

In [ ]:
graph_path = save_graph_visualization(
    prepared.graph, project_root / "outputs/tracking/notebook_conflict_graph.png"
)
display(Image(filename=str(graph_path)))

In [ ]:
comparison = tracker.compare_prepared(prepared, (classical, quantum))
common_columns = (
    "problem_id", "input_fingerprint", "backend", "status",
    "feasible", "selected_ids", "objective", "runtime_seconds",
)
[{key: row[key] for key in common_columns} for row in comparison.rows()]

A quantum `embedding_error` or `unsupported_size` is part of the common result contract, not a hidden classical fallback. We explicitly choose a successful run below; changing the name to `neutral_atom_qutip` is the only change needed to advance with the simulated QC result.

In [ ]:
advanced = tracker.advance(prepared, comparison.run("classical_exact"))
[(track.track_id, track.position, track.posterior_probability) for track in advanced.tracks]

## Optional detector data adapter

The same interface accepts stage-one cell detections. This cell reads only the versioned event table and demonstrates the adapter; it does not run the large graph through the state-vector simulator.

In [ ]:
from neutral_atom_mht.io import read_detections
from neutral_atom_mht.tracking import observations_from_detections

events_path = project_root / "artifacts/detection/sequence_01/detections.csv"
events = read_detections(events_path) if events_path.exists() else ()
frame_zero = tuple(event for event in events if event.frame == 0)[:3]
observations_from_detections(frame_zero)